# Steerling-8B: Concept Suppresion

**Concept Suppresion** suppresses a target concept during generation, with no weight updates. Steerling does this at inference time by intervening on its concept representations.

The public Suppresion recipe is `injection_relu`, which combines two mechanisms. **Negative injection** adds the negated concept direction to the residual stream (the running hidden state) at the deeper layers, pushing generation away from the concept. The **ReLU logit mask** then subtracts the concept's positive alignment from the output logits, removing the specific tokens the concept promotes without boosting unrelated ones.

This notebook generates twice on the same prompt, a reference and an suppressed run, then checks that the target concept's activation drops.

**Requirements:** an interpretable Steerling checkpoint and a GPU with >= 18 GB VRAM.

## 1. Load the model

Suppresion needs the **interpretable** model, which exposes the concept heads the intervention acts on. We load via HuggingFace `AutoModel` and wrap it in `SteerlingGenerator`.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
from steerling import SteerlingGenerator, GenerationConfig, SteeringConfig

model = AutoModel.from_pretrained("guidelabs/steerling-8b-instruct", trust_remote_code=True, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained("guidelabs/steerling-8b-instruct", trust_remote_code=True)
generator = SteerlingGenerator.from_model(model, tokenizer, device="cuda")

## 2. Choose a concept and prime the prompt

Set the target concept and a prompt. To make suppression visible, we **prime** the prompt toward the concept, the same way the steering evaluation does: prepend a short description so an unsteered model would lean into the concept. suppressing should pull the generation back out.

Set `CONCEPT_ID`, `CONCEPT_LABEL`, and `CONCEPT_DESCRIPTION` to a concept from your concept table.

In [ ]:
# --- target concept (replace with one from your concept table) ---
CONCEPT_ID = 20353
CONCEPT_LABEL = "Books and publishing"
CONCEPT_DESCRIPTION = "Topic about (publish, publisher, publication, Publishing, .pub, _publish), industry infrastructure (imprint, ISBN, Kindle, ePub, Penguin, Harper, Barnes & Noble, Ingram...)"

# --- prompt, primed toward the concept (as a chat turn) ---
BASE_PROMPT = "Tell me something interesting."
USER_CONTENT = f"We are discussing about {CONCEPT_LABEL}: {CONCEPT_DESCRIPTION}\n{BASE_PROMPT}"

messages = [
    {"role": "system", "content": "You are a helpful AI assistant. Be concise."},
    {"role": "user", "content": USER_CONTENT},
]
REFERENCE_PROMPT = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# --- generation budget ---
MAX_NEW_TOKENS = 128
STEPS = 128

# --- suppression strength ---
# mai_lm_target is the raw injection alpha. Negative suppresses.
# adjust to get the best suppression + text quality
ALPHA = -7.0
# inject_layer=None resolves to n_layers // 2 inside the generator.
INJECT_LAYER = None

# stop at the end of the assistant turn
EOT_ID = tokenizer.convert_tokens_to_ids("<|eot_id|>")

print(REFERENCE_PROMPT)

## 3. Reference generation
As we put the topic into the prompt, the model will talk about the target concept.
First generate with no steering. This is the baseline the suppressed run is compared against. Reference sampling uses `temperature=1.2`, `top_p=0.8`, `repetition_penalty=1.1`.

In [ ]:
ref_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS,
    steps=STEPS,
    temperature=1.2,
    top_p=0.8,
    repetition_penalty=1.1,
    seed=0,
    stop_tokens=[EOT_ID],
)

reference = generator.generate_full(REFERENCE_PROMPT, ref_config)
print(reference.text)

## 4. Suppress the concept (`injection_relu`)

Steered sampling uses `temperature=0.9`, `top_p=0.7`, `repetition_penalty=1.5`.

In [ ]:
steered_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS,
    steps=STEPS,
    temperature=0.9,
    top_p=0.7,
    repetition_penalty=1.5,
    seed=0,
    stop_tokens=[EOT_ID],
)

suppress = SteeringConfig.injection_relu(
    concept_ids=CONCEPT_ID,
    mai_lm_target=ALPHA,
    inject_layer=INJECT_LAYER,
)

suppressed = generator.generate_steered(REFERENCE_PROMPT, steered_config, suppress)
print(suppressed.text)